In [7]:
%pip install duckdb pandas

   ---------------------------------------- 0.0/13.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/13.1 MB ? eta -:--:--
    --------------------------------------- 0.3/13.1 MB ? eta -:--:--
    --------------------------------------- 0.3/13.1 MB ? eta -:--:--
    --------------------------------------- 0.3/13.1 MB ? eta -:--:--
   - -------------------------------------- 0.5/13.1 MB 386.9 kB/s eta 0:00:33
   - -------------------------------------- 0.5/13.1 MB 386.9 kB/s eta 0:00:33
   - -------------------------------------- 0.5/13.1 MB 386.9 kB/s eta 0:00:33
   -- ------------------------------------- 0.8/13.1 MB 425.2 kB/s eta 0:00:30
   -- ------------------------------------- 0.8/13.1 MB 425.2 kB/s eta 0:00:30
   -- ------------------------------------- 0.8/13.1 MB 425.2 kB/s eta 0:00:30
   --- ------------------------------------ 1.0/13.1 MB 419.1 kB/s eta 0:00:29
   --- ------------------------------------ 1.0/13.1 MB 419.1 kB/s eta 0:00:29
   --- -----------

In [8]:
import os
import duckdb
import pandas as pd
from datetime import datetime

In [9]:
import glob

# Buscar todos los archivos parquet en la carpeta bronze
archivos_bronze = glob.glob("../Data/Bronze/weather_raw_*.parquet")

if not archivos_bronze:
    raise FileNotFoundError("No se encontraron archivos en la capa Bronze.")

# Ordenar por nombre (la fecha en el nombre nos asegura que el último es el más nuevo)
ultimo_archivo = max(archivos_bronze)
print(f"Archivo detectado para procesar: {ultimo_archivo}")

Archivo detectado para procesar: ../Data/Bronze\weather_raw_20260625_1208.parquet


In [10]:
# Conectamos a una base de datos DuckDB en memoria (o la creamos si no existe)
con = duckdb.connect()

# Query SQL para transformar los datos directamente desde el Parquet
query_silver = f"""
    SELECT 
        CAST(time AS TIMESTAMP) AS fecha_hora,
        CAST(temperature_2m AS FLOAT) AS temperatura_celsius,
        CAST(relative_humidity_2m AS INT) AS humedad_porcentaje,
        CAST(rain AS FLOAT) AS lluvia_mm,
        CAST(precipitation AS FLOAT) AS precipitacion_acumulada_mm,
        CAST(ingested_at AS TIMESTAMP) AS fecha_ingesta
    FROM read_parquet('{ultimo_archivo}')
    WHERE time IS NOT NULL
"""

# Ejecutar la query y guardar el resultado en un DataFrame de Pandas
df_silver = con.execute(query_silver).df()

print("Muestra de los datos normalizados en Silver:")
print(df_silver.head())

Muestra de los datos normalizados en Silver:
           fecha_hora  temperatura_celsius  humedad_porcentaje  lluvia_mm  \
0 2026-06-25 00:00:00                 10.4                  80        0.0   
1 2026-06-25 01:00:00                 10.2                  81        0.0   
2 2026-06-25 02:00:00                  9.7                  85        0.0   
3 2026-06-25 03:00:00                  9.3                  90        0.0   
4 2026-06-25 04:00:00                  8.5                  92        0.0   

   precipitacion_acumulada_mm              fecha_ingesta  
0                         0.0 2026-06-25 12:04:47.496514  
1                         0.0 2026-06-25 12:04:47.496514  
2                         0.0 2026-06-25 12:04:47.496514  
3                         0.0 2026-06-25 12:04:47.496514  
4                         0.0 2026-06-25 12:04:47.496514  


In [12]:
# Definimos la ruta asegurándonos de usar minúsculas para mantener consistencia
ruta_silver = "../Data/silver/weather_clean.parquet"

# Usamos el parámetro por nombre 'exist_ok=True' de forma explícita
os.makedirs("../Data/silver/", exist_ok=True)

# Volvemos a asegurar que si el archivo parquet ya existía, DuckDB lo reemplace
# Para eso usamos la opción OVERWRITE dentro de DuckDB (o borramos el archivo previo)
if os.path.exists(ruta_silver):
    os.remove(ruta_silver)

# DuckDB escribe el DataFrame directamente a Parquet
con.execute(f"COPY (SELECT * FROM df_silver) TO '{ruta_silver}' (FORMAT PARQUET);")

print(f"¡Capa Silver procesada y guardada con éxito en: {ruta_silver}!")

¡Capa Silver procesada y guardada con éxito en: ../Data/silver/weather_clean.parquet!
